In [23]:
## Импортирую библиотеки для анализа данных
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

In [24]:
## Загружаю данные
df = pd.read_csv('data/TMDB_all_movies.csv', nrows=5000)

In [25]:
## Проверяю длительность фильмов на присутствие аномальных значений
df["runtime"].describe()

count    5000.000000
mean      102.792600
std        27.781691
min         0.000000
25%        90.000000
50%       100.000000
75%       115.000000
max       335.000000
Name: runtime, dtype: float64

In [ ]:
  ## Проверяю кол-во значений в столбце с рейтингами фильмов
df["certification_us"].value_counts(dropna=False)

certification_us
NaN      2202
R        1228
PG-13     653
PG        450
NR        350
G          89
NC-17      28
Name: count, dtype: int64

In [27]:
## Смотрю на первые 10 значений в столбцах с жанрами и датой релиза, а также их типы данных. 
print(df["genres"].head(10))
print(df["release_date"].head(10))
print(df.dtypes[["genres", "release_date"]])

0                 Drama, Romance, Crime
1                        Drama, Romance
2                                Comedy
3        Action, Crime, Drama, Thriller
4                           Documentary
5                                 Drama
6    Adventure, Action, Science Fiction
7          Animation, Family, Adventure
8                Comedy, Drama, Romance
9                                 Drama
Name: genres, dtype: str
0    1988-10-21
1    1986-10-17
2    1995-12-09
3    1993-10-15
4    2006-01-01
5    2004-09-02
6    1977-05-25
7    2003-05-30
8    1994-06-23
9    1999-09-15
Name: release_date, dtype: str
genres          str
release_date    str
dtype: object


In [28]:
## Смотрю долю нулей в budget и revenue
print(f"Нулевой бюджет: {(df['budget'] == 0).mean() * 100:.2f}%")
print(f"Нулевые сборы: {(df['revenue'] == 0).mean() * 100:.2f}%")
print(f"Есть и бюджет, и сборы: {((df['budget'] > 0) & (df['revenue'] > 0)).sum()}")

Нулевой бюджет: 48.82%
Нулевые сборы: 46.84%
Есть и бюджет, и сборы: 2292


In [29]:
## Проверяю, отсортирован ли файл по известности: смотрю на vote_count в первых строках
print(df["vote_count"].describe())
print(df[["title", "vote_count"]].head(5))

count     5000.000000
mean      1514.932400
std       3263.760691
min          0.000000
25%         23.000000
50%        265.000000
75%       1464.250000
max      36828.000000
Name: vote_count, dtype: float64
                              title  vote_count
0                             Ariel       382.0
1               Shadows in Paradise       448.0
2                        Four Rooms      2901.0
3                    Judgment Night       387.0
4  Life in Loops (A Megacities RMX)        34.0


In [30]:
## Проверяю, идут ли id по возрастанию и какие годы релиза в первых 5000 строках
print(df["id"].head(10).tolist())
print(df["id"].is_monotonic_increasing)
print(pd.to_datetime(df["release_date"]).dt.year.describe())

[2, 3, 5, 6, 8, 9, 11, 12, 13, 14]
True
count    4996.000000
mean     1988.020616
std        20.137271
min      1895.000000
25%      1975.000000
50%      1996.000000
75%      2004.000000
max      2026.000000
Name: release_date, dtype: float64


In [31]:
## Подключаюсь к базе movies: параметры (пароль) читаю из .env
load_dotenv()
url = (f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
       f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}")
engine = create_engine(url)

with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database()")).scalar())

movies


In [ ]:
## Загружаю CSV в таблицу movies частями по 50 000 строк: привожу типы и пишу в базу
## Колонку genres здесь не грузим, жанры уйдут в отдельные таблицы
cols = [
    "id", "title", "status", "release_date", "runtime", "budget", "revenue", "certification_us", "imdb_rating", "imdb_votes", "vote_count", "director", "writers", "original_language", "production_countries", "imdb_id", "genres",
]

int_cols = ["runtime", "budget", "revenue", "imdb_votes", "vote_count"]

reader = pd.read_csv("data/TMDB_all_movies.csv", usecols=cols,
                     chunksize=50_000)

for chunk in reader:
    ## Даты: битые значения станут NULL
    chunk["release_date"] = pd.to_datetime(chunk["release_date"], errors="coerce")
    ## Целые с пропусками: Int64 (с большой буквы) допускает NULL
    for c in int_cols:
        chunk[c] = pd.to_numeric(chunk[c], errors="coerce").round().astype("Int64")
    ## Рейтинг: дробное число
    chunk["imdb_rating"] = pd.to_numeric(chunk["imdb_rating"], errors="coerce")
    # Жанры в movies не нужны
    chunk = chunk.drop(columns="genres")
    chunk.to_sql("movies", engine, if_exists="append", index=False,
                 method="multi", chunksize=5000)
    print("загружено", len(chunk))

загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 50000
загружено 553


In [ ]:
## Считаю строки в CSV частями, чтобы сверить с числом строк в базе
total = sum(len(c) for c in pd.read_csv("data/TMDB_all_movies.csv", usecols=["id"], chunksize=100_000))
print(total)

1250553


In [ ]:
# Тест на малом объёме (1000 строк): готовлю пары «фильм, жанр», в базу пока не пишу
reader_1 = pd.read_csv("data/TMDB_all_movies.csv", usecols=["id", "genres"],
                       chunksize=50_000, nrows=1000)

for chunk in reader_1:
    ## Убираю фильмы без жанров
    chunk = chunk.dropna(subset=["genres"])
    ## Строку «Drama, Romance» превращаю в список ["Drama", "Romance"]
    chunk["genres"] = chunk["genres"].str.split(", ")
    ## Один жанр на строку: фильм с тремя жанрами даст три строки
    chunk = chunk.explode("genres")
    ## Убираю пробелы по краям, чтобы «Action» и « Action» не считались разными жанрами
    chunk["genres"] = chunk["genres"].str.strip()

## Проверяю вид данных и список уникальных жанров (без NaN и пробелов)
print(chunk.head(10))
print(chunk["genres"].unique())

   id    genres
0   2     Drama
0   2   Romance
0   2     Crime
1   3     Drama
1   3   Romance
2   5    Comedy
3   6    Action
3   6     Crime
3   6     Drama
3   6  Thriller
<StringArray>
[          'Drama',         'Romance',           'Crime',          'Comedy',
          'Action',        'Thriller',     'Documentary',       'Adventure',
 'Science Fiction',       'Animation',          'Family',         'Mystery',
          'Horror',         'Fantasy',             'War',           'Music',
         'Western',         'History',        'TV Movie']
Length: 19, dtype: str


In [36]:
## Собираю пары (фильм, жанр) из всего CSV частями; в базу пока не пишу
parts = []
for chunk in pd.read_csv("data/TMDB_all_movies.csv", usecols=["id", "genres"], chunksize=50_000):
    ## Убираю фильмы без жанров
    chunk = chunk.dropna(subset=["genres"])
    ## Строка жанров -> список -> по одному жанру на строку
    chunk["genres"] = chunk["genres"].str.split(", ")
    chunk = chunk.explode("genres")
    chunk["genres"] = chunk["genres"].str.strip()
    parts.append(chunk)

## Склеиваю куски; вывожу число пар и число уникальных жанров (ожидаю 19)
pairs = pd.concat(parts, ignore_index=True)
print(len(pairs), pairs["genres"].nunique())

1410286 19


In [37]:
## Заполняю справочник genres: каждый жанр записываю один раз
## Беру из пар только колонку с названиями жанров
genre_names = pairs["genres"].drop_duplicates()

## Превращаю в таблицу с колонкой genre_name, чтобы название совпало с колонкой в базе
genres_df = genre_names.to_frame(name="genre_name")

## Записываю в таблицу genres; genre_id не передаю, его создаёт SERIAL
genres_df.to_sql("genres", engine, if_exists="append", index=False)

## Проверяю: должно быть 19 жанров
print(len(genres_df))

19


In [38]:
## Заполняю movie_genres: беру genre_id из базы, связываю с фильмами и записываю пары
## Читаю справочник жанров обратно из базы, чтобы получить genre_id
genres_db = pd.read_sql("SELECT genre_id, genre_name FROM genres", engine)

## Присоединяю genre_id к парам, оставляю две колонки и переименую id в movie_id
movie_genres = (
    pairs.merge(genres_db, left_on="genres", right_on="genre_name")
    [["id", "genre_id"]]
    .rename(columns={"id": "movie_id"})
    .drop_duplicates()
)

## Проверяю число пар перед записью (ожидаю 1 410 286)
print(len(movie_genres))

## Записываю в movie_genres частями
movie_genres.to_sql("movie_genres", engine, if_exists="append", index=False,
                    method="multi", chunksize=5000)

1410171


1410171

In [39]:
## Проверяю, сколько дублей (фильм, жанр) было в исходных парах
print(pairs.duplicated().sum())

115
